# Task 4: Evaluation of Recommender System

## a) Evaluate the performance of your content-based and collaborative filtering recommender systems using a metric such as Mean Squared Error (MSE) or  Root Mean Squared Error (RMSE)

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, KNNBasic, SVD
from surprise.model_selection import cross_validate
import warnings
warnings.filterwarnings('ignore')

# Load the data
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

### Evaluate performance on content-based filtering recommender systems

In [2]:
def evaluate_content_based():
    # Merge data to get movie titles with ratings
    merged_data = pd.merge(ratings, movies, on='movieId')

    # Split data into train and test sets
    train_data, test_data = train_test_split(merged_data, test_size=0.2, random_state=42)

    # Create TF-IDF matrix for movie genres
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(movies['genres'].fillna(''))

    # Calculate cosine similarity between movies
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    # Create movie index mapping
    movie_indices = pd.Series(movies.index, index=movies['movieId']).drop_duplicates()

    def predict_rating_content_based(user_ratings, movie_id):
        """Predict rating based on weighted average of similar movies"""
        if movie_id not in movie_indices:
            return ratings['rating'].mean()  # Return average rating if movie not found

        idx = movie_indices[movie_id]
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:11]  # Top 10 similar movies (excluding itself)

        movie_indices_sim = [i[0] for i in sim_scores]
        sim_values = [i[1] for i in sim_scores]

        # Get user's ratings for similar movies
        similar_movie_ids = movies.iloc[movie_indices_sim]['movieId']
        user_ratings_sim = user_ratings[user_ratings['movieId'].isin(similar_movie_ids)]

        if len(user_ratings_sim) == 0:
            return ratings['rating'].mean()

        # Calculate weighted average
        weighted_sum = 0
        total_similarity = 0

        for _, row in user_ratings_sim.iterrows():
            sim_idx = movie_indices[row['movieId']]
            similarity = cosine_sim[idx][sim_idx]
            weighted_sum += row['rating'] * similarity
            total_similarity += similarity

        if total_similarity == 0:
            return ratings['rating'].mean()

        return weighted_sum / total_similarity

    # Make predictions on test set
    predictions = []
    actual_ratings = []

    print(f"Evaluating Content-Based Filtering for test data {test_data.shape[0]} samples...")
    for _, row in test_data.iterrows():
        user_id = row['userId']
        movie_id = row['movieId']
        actual_rating = row['rating']

        # Get user's ratings from training data
        user_train_ratings = train_data[train_data['userId'] == user_id]

        predicted_rating = predict_rating_content_based(user_train_ratings, movie_id)

        predictions.append(predicted_rating)
        actual_ratings.append(actual_rating)

    # Calculate evaluation metrics
    mse = mean_squared_error(actual_ratings, predictions)
    rmse = np.sqrt(mse)

    return mse, rmse, predictions, actual_ratings

# Evaluate content-based filtering
cb_mse, cb_rmse, cb_predictions, cb_actual = evaluate_content_based()
print("\nContent-Based Filtering Results:")
print(f"Content-Based Filtering - MSE: {cb_mse:.4f}, RMSE: {cb_rmse:.4f}")

Evaluating Content-Based Filtering for test data 20168 samples...

Content-Based Filtering Results:
Content-Based Filtering - MSE: 1.1948, RMSE: 1.0931


### Evaluate performance on collaborative filtering recommender

In [3]:
def evaluate_collaborative_filtering():
    # Prepare data for Surprise library
    reader = Reader(rating_scale=(0.5, 5.0))
    data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

    # User-based collaborative filtering
    sim_options = {'name': 'cosine', 'user_based': True}
    user_based = KNNBasic(sim_options=sim_options)

    # Item-based collaborative filtering
    sim_options = {'name': 'cosine', 'user_based': False}
    item_based = KNNBasic(sim_options=sim_options)

    # Matrix Factorization (SVD)
    svd = SVD()

    # Evaluate all models using cross-validation
    print("Evaluating Collaborative Filtering models...")

    # cross_validate returns a dict with test_rmse and test_mse among other metrics
    user_based_results = cross_validate(user_based, data, measures=['RMSE', 'MSE'], cv=5, verbose=False)
    item_based_results = cross_validate(item_based, data, measures=['RMSE', 'MSE'], cv=5, verbose=False)
    svd_results = cross_validate(svd, data, measures=['RMSE', 'MSE'], cv=5, verbose=False)

    return {
        'user_based': {
            'rmse': np.mean(user_based_results['test_rmse']),
            'mse': np.mean(user_based_results['test_mse'])
        },
        'item_based': {
            'rmse': np.mean(item_based_results['test_rmse']),
            'mse': np.mean(item_based_results['test_mse'])
        },
        'svd': {
            'rmse': np.mean(svd_results['test_rmse']),
            'mse': np.mean(svd_results['test_mse'])
        }
    }

# Evaluate collaborative filtering
cf_results = evaluate_collaborative_filtering()

print("\nCollaborative Filtering Results:")
print(f"User-Based - MSE: {cf_results['user_based']['mse']:.4f}, RMSE: {cf_results['user_based']['rmse']:.4f}")
print(f"Item-Based - MSE: {cf_results['item_based']['mse']:.4f}, RMSE: {cf_results['item_based']['rmse']:.4f}")
print(f"SVD (Matrix Factorization) - MSE: {cf_results['svd']['mse']:.4f}, RMSE: {cf_results['svd']['rmse']:.4f}")

Evaluating Collaborative Filtering models...
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.

Collaborative Filtering Results:
User-Based - MSE: 0.9465, RMSE: 0.9729
Item-Based - MSE: 0.9537, RMSE: 0.9766
SVD (Matrix Factorization) - MSE: 0.7621, RMSE: 0.8730


### Compare all recommender systems' performance

In [4]:
def compare_all_methods():
    print("=" * 60)
    print("Recommender System Comparison")
    print("=" * 60)

    # Content-based results
    print(f"Content-Based Filtering:")
    print(f"  MSE:  {cb_mse:.4f}")
    print(f"  RMSE: {cb_rmse:.4f}")
    print("  Analysis: Content-based methods rely on movie metadata (genres) and work well if users have rated enough items. "
          "However, they struggle with cold-start problems for new users and movies.")
    print()

    # Collaborative filtering results
    print("Collaborative Filtering:")
    print(f"  User-Based:")
    print(f"    MSE:  {cf_results['user_based']['mse']:.4f}")
    print(f"    RMSE: {cf_results['user_based']['rmse']:.4f}")
    print("    Analysis: Good when there are many overlapping ratings between users. Performance drops with sparse data "
          "or new users who haven’t rated many movies.")
    print()

    print(f"  Item-Based:")
    print(f"    MSE:  {cf_results['item_based']['mse']:.4f}")
    print(f"    RMSE: {cf_results['item_based']['rmse']:.4f}")
    print("    Analysis: More stable than user-based because item similarities don’t change often. Works well when items "
          "have many ratings but can be weak for new or rarely-rated movies.")
    print()

    print(f"  SVD (Matrix Factorization):")
    print(f"    MSE:  {cf_results['svd']['mse']:.4f}")
    print(f"    RMSE: {cf_results['svd']['rmse']:.4f}")
    print("    Analysis: Captures hidden patterns and is usually the strongest for large datasets. Requires tuning factors "
          "and can be computationally heavier.")
    print()

    # Find best method
    all_rmse = {
        'Content-Based': cb_rmse,
        'User-Based CF': cf_results['user_based']['rmse'],
        'Item-Based CF': cf_results['item_based']['rmse'],
        'SVD': cf_results['svd']['rmse']
    }

    best_method = min(all_rmse, key=all_rmse.get)
    print(f"Best performing method: {best_method} (RMSE: {all_rmse[best_method]:.4f})")
    print("Note: Lower RMSE means better accuracy, but consider scalability, cold-start handling, and explainability when "
          "choosing a production-ready method.")

# Run comparison
compare_all_methods()


Recommender System Comparison
Content-Based Filtering:
  MSE:  1.1948
  RMSE: 1.0931
  Analysis: Content-based methods rely on movie metadata (genres) and work well if users have rated enough items. However, they struggle with cold-start problems for new users and movies.

Collaborative Filtering:
  User-Based:
    MSE:  0.9465
    RMSE: 0.9729
    Analysis: Good when there are many overlapping ratings between users. Performance drops with sparse data or new users who haven’t rated many movies.

  Item-Based:
    MSE:  0.9537
    RMSE: 0.9766
    Analysis: More stable than user-based because item similarities don’t change often. Works well when items have many ratings but can be weak for new or rarely-rated movies.

  SVD (Matrix Factorization):
    MSE:  0.7621
    RMSE: 0.8730
    Analysis: Captures hidden patterns and is usually the strongest for large datasets. Requires tuning factors and can be computationally heavier.

Best performing method: SVD (RMSE: 0.8730)
Note: Lower RMSE m

## b) Test the performance on a subset of the movie dataset and analyse how well the system is performing.

### Evaluate recommender systems with 5000 dataset

In [5]:
# Subset: first 5000 ratings
ratings_subset = ratings.head(5000)

# Update the global ratings DataFrame temporarily
ratings_backup = ratings.copy()
ratings = ratings_subset

# Evaluate on subset
cb_mse, cb_rmse, _, _ = evaluate_content_based()
cf_results = evaluate_collaborative_filtering()

# Restore the original ratings DataFrame
ratings = ratings_backup

Evaluating Content-Based Filtering for test data 1000 samples...
Evaluating Collaborative Filtering models...
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.


### Compare all recommenders' performance

In [6]:
compare_all_methods()

Recommender System Comparison
Content-Based Filtering:
  MSE:  1.1467
  RMSE: 1.0708
  Analysis: Content-based methods rely on movie metadata (genres) and work well if users have rated enough items. However, they struggle with cold-start problems for new users and movies.

Collaborative Filtering:
  User-Based:
    MSE:  1.3872
    RMSE: 1.1775
    Analysis: Good when there are many overlapping ratings between users. Performance drops with sparse data or new users who haven’t rated many movies.

  Item-Based:
    MSE:  1.0243
    RMSE: 1.0119
    Analysis: More stable than user-based because item similarities don’t change often. Works well when items have many ratings but can be weak for new or rarely-rated movies.

  SVD (Matrix Factorization):
    MSE:  0.9115
    RMSE: 0.9542
    Analysis: Captures hidden patterns and is usually the strongest for large datasets. Requires tuning factors and can be computationally heavier.

Best performing method: SVD (RMSE: 0.9542)
Note: Lower RMSE m

## c) Provide the code and analysis of the recommender system’s performance.

### Full Dataset vs. Subset

-   **Full dataset:** More ratings and users mean better similarity
    measures and lower error metrics (RMSE, MSE). Predictions are
    generally more accurate.
-   **Subset:** With fewer ratings, there’s less overlap and higher
    sparsity. RMSE often increases because models have less context to
    learn from.

### Best Recommender

-   **Content-Based Filtering:** Good for cold-start scenarios or when
    item metadata is rich. However, it usually lags behind collaborative
    methods in accuracy.
-   **Collaborative Filtering:** User- and item-based KNN perform
    decently, but matrix factorization (SVD) tends to perform best,
    capturing hidden user and item patterns.
-   **Recommendation:** **SVD is generally the best-performing method**
    when enough data is available. Content-based is a backup when
    dealing with new users or items.